# 07 - Agente: herramientas del modelo, dominio y verificacion con partidas reales

Rama: `07MultiAgentIntegration`. Este cuaderno **no entrena ni recalibra** nada: prepara lo que el agente necesita y comprueba que consume el modelo final tal como se evaluo en la Etapa 2.

| Parte | Que hace | Publica en Kaggle |
|---|---|---|
| 1 | Perfil del dominio de entrenamiento (train de A): rangos por caracteristica, cartas vistas y cortes de etapa | `agente/dominio.json` |
| 2 | Repeticiones reales de ejemplo (partidas de `test` de A del 22 de julio) | `agente/ejemplos/` |
| 3 | Verificacion: el agente da **la misma probabilidad** que `PredictorVictoria` en cada estado real | nada |
| 4 | Demostracion de las herramientas, casos limite y registro de invocaciones | nada |

El uso con Claude Desktop o Claude Code (local) esta al final.

In [25]:
# PREPARACION COMPLETA DEL CUADERNO (rama 07MultiAgentIntegration)
# PROPOSITO: dejar el servidor listo: repo en la rama 07, librerias fijadas, tablas del
# dataset del equipo en /content/datos y los archivos del modelo final en /content/resultados.
# ESPERADO: la linea de nucleos/RAM, "Tablas listas | caracteristicas: 62" y
# "Modelo final y metricas de test listos".
import json
import subprocess
import sys
import time
from pathlib import Path

REPO = "Adriel23456/Applied-AI-Agent-Project"
BRANCH = "07MultiAgentIntegration"
REPO_DIR = Path("/content/repo")
if not REPO_DIR.exists():
    !git clone -b {BRANCH} https://github.com/{REPO}.git /content/repo
else:
    !git -C /content/repo fetch origin {BRANCH}
    !git -C /content/repo checkout {BRANCH}
    !git -C /content/repo pull --ff-only origin {BRANCH}
%cd /content/repo
!pip install -q -r requirements.txt
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for nombre in [m for m in list(sys.modules) if m.split(".")[0] in ("ptcg_wp", "ptcg_agente")]:
    del sys.modules[nombre]

from ptcg_wp.colab import recursos, setup_kaggle
RECURSOS = recursos()
setup_kaggle()

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from sklearn import __version__ as sklearn_version
assert sklearn_version == "1.7.2", (f"Se requiere scikit-learn 1.7.2; version cargada: {sklearn_version}. "
                                    "Reinicia el runtime y ejecuta esta celda de nuevo.")
from ptcg_wp import store
from ptcg_wp.features import feature_columns
from ptcg_wp.ingest import load_days
from ptcg_wp.predictor import FICHA_EXPORTADA, MODELO_EXPORTADO
from ptcg_agente import artefactos

DATA_DIR = store.DATOS_DIR
RESULTADOS = store.RESULTADOS_DIR
DATASET = store.load_dataset_id()
store.fetch(DATASET, DATA_DIR, store.expected_files(load_days("configs/dias.yaml")))
schema = pq.ParquetFile(sorted(DATA_DIR.glob("estados_*.parquet"))[0]).schema.names
assert len(feature_columns(schema)) == 62
print("Tablas listas | caracteristicas: 62")

BASE = [MODELO_EXPORTADO, FICHA_EXPORTADA, *artefactos.METRICAS_TEST, artefactos.VEREDICTO]
assert store.listo(BASE, RESULTADOS), "Faltan el modelo final o las metricas de test en Kaggle"
print("Modelo final y metricas de test listos")

COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
DIRTY = bool(subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True).strip())
print("rama:", BRANCH, "| commit:", COMMIT[:7], "| cambios locales:", DIRTY)

remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 22 (delta 11), reused 22 (delta 11), pack-reused 0 (from 0)
Unpacking objects: 100% (22/22), 18.95 KiB | 1.89 MiB/s, done.
From https://github.com/Adriel23456/Applied-AI-Agent-Project
 * branch            07MultiAgentIntegration -> FETCH_HEAD
   f0c4d9c..428d028  07MultiAgentIntegration -> origin/07MultiAgentIntegration
Already on '07MultiAgentIntegration'
Your branch is behind 'origin/07MultiAgentIntegration' by 2 commits, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/Adriel23456/Applied-AI-Agent-Project
 * branch            07MultiAgentIntegration -> FETCH_HEAD
Updating f0c4d9c..428d028
Fast-forward
 docx/05AmbienteDeDesarrollo/GuiaDelAgente.md |  68 ++-
 notebooks/07_agente.ipynb                    | 705 ++++++++++++++++++++++++++-
 requirements.txt                             |   1 +
 

In [26]:
# PRUEBAS DEL AGENTE
# PROPOSITO: correr las pruebas automaticas del agente (herramientas, verificador, artefactos,
# registro y servidor MCP) y del modelo exportado, con datos sinteticos y sin Kaggle.
# ESPERADO: "25 passed" y la linea "OK: pruebas del agente aprobadas".
resultado = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_agente.py",
                            "tests/test_predictor.py"], cwd=REPO_DIR, capture_output=True, text=True)
print(resultado.stdout[-2000:], resultado.stderr[-2000:])
assert resultado.returncode == 0, "fallaron pruebas del agente: revisa la salida"
print("OK: pruebas del agente aprobadas")

..........................                                               [100%]
26 passed in 4.74s
 
OK: pruebas del agente aprobadas


## Parte 1 - Perfil del dominio de entrenamiento

`revisar_dominio` y las advertencias del agente comparan cada entrada con lo que el modelo vio al entrenar. El perfil se calcula **solo con las filas `train` del escenario A**, que son las del modelo final. Guarda, por caracteristica, minimo, maximo y percentiles 0.5 y 99.5; las cartas que aparecieron como Pokemon activo; y los cortes de etapa.

In [27]:
# PARTE 1 - PERFIL DEL DOMINIO (agente/dominio.json)
# PROPOSITO: construir el perfil con el train de A y publicarlo; si ya esta en Kaggle, solo se baja.
# ESPERADO: ~4.4 millones de filas de train, cortes de etapa (5.0, 9.0) y unas 360 cartas activas.
from ptcg_agente import dominio
from ptcg_wp.data import load_scenario
from ptcg_wp.predictor import PredictorVictoria
from ptcg_wp.splits import stage_cutoffs

FEATURES_MODELO = PredictorVictoria.cargar(RESULTADOS / MODELO_EXPORTADO).features


def construir_dominio():
    estados, feats, split, _ = load_scenario(DATA_DIR, "A", include_cards=True)
    train = estados.loc[split == "train"]
    features = [f for f in feats if f not in dominio.CARTAS]
    assert features == list(FEATURES_MODELO), "las caracteristicas no coinciden con las del modelo final"
    perfil = dominio.construir_perfil(train, features, stage_cutoffs(train["turn"].to_numpy()))
    perfil.update(commit=COMMIT, fecha_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
    destino = RESULTADOS / artefactos.DOMINIO
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_text(json.dumps(perfil, indent=2), encoding="utf-8")


store.asegurar([artefactos.DOMINIO], construir_dominio,
               "agente: perfil del dominio de entrenamiento (train de A)", RESULTADOS)
PERFIL = json.loads((RESULTADOS / artefactos.DOMINIO).read_text(encoding="utf-8"))
print(f"filas train: {PERFIL['n_filas']:,} | partidas: {PERFIL['n_partidas']:,} | "
      f"cortes de etapa: {PERFIL['cortes_etapa']} | cartas activas vistas: {len(PERFIL['cartas_activas'])}")
display(pd.DataFrame(PERFIL["rangos"]).T.loc[["game_turn", "own_deck", "own_prizes", "diff_prizes",
                                              "own_active_hp", "opp_discard"]])

ya local: agente/dominio.json
=> Todo estaba en Kaggle: se bajo y NO se entreno.
filas train: 4,357,908 | partidas: 33,066 | cortes de etapa: [5.0, 9.0] | cartas activas vistas: 362


,min,max,p005,p995
game_turn,1.0,285.0,1.0,54.0
own_deck,0.0,50.0,0.0,46.0
own_prizes,1.0,6.0,1.0,6.0
diff_prizes,-5.0,5.0,-4.0,5.0
own_active_hp,-1.0,440.0,-1.0,440.0
opp_discard,0.0,57.0,0.0,41.0


## Parte 2 - Repeticiones reales de ejemplo

Cuatro partidas de `test` del escenario A del 22 de julio, de duraciones distintas. El modelo nunca las vio. Se baja solo el JSON de cada partida del dataset publico (no el dia completo) y se publican en el dataset del equipo para que cualquiera pueda probar el agente localmente.

In [28]:
# PARTE 2 - EJEMPLOS REALES (agente/ejemplos/partida_<id>.json)
# PROPOSITO: elegir 4 partidas de test de A (una por cuartil de duracion) y publicarlas.
# ESPERADO: 4 rutas agente/ejemplos/partida_<id>.json y su duracion en pasos.
from ptcg_agente import ejemplos

DIA_EJEMPLOS = "2026-07-22"
IDS = ejemplos.elegir(DATA_DIR, DIA_EJEMPLOS, n=4)
RUTAS_EJEMPLOS = [ejemplos.ruta(g) for g in IDS]


def bajar_ejemplos():
    for g in IDS:
        print("bajada:", ejemplos.bajar(DIA_EJEMPLOS, g, RESULTADOS))


store.asegurar(RUTAS_EJEMPLOS, bajar_ejemplos, "agente: repeticiones reales de ejemplo (test de A)", RESULTADOS)
partidas = pd.read_parquet(DATA_DIR / f"partidas_{DIA_EJEMPLOS}.parquet", columns=["game_id", "n_steps", "winner"])
elegidas = pd.DataFrame({"game_id": IDS, "archivo": RUTAS_EJEMPLOS}).merge(partidas, on="game_id")
display(elegidas[["game_id", "n_steps", "winner", "archivo"]])

ya local: agente/ejemplos/partida_87542385.json
ya local: agente/ejemplos/partida_87406956.json
ya local: agente/ejemplos/partida_87363568.json
ya local: agente/ejemplos/partida_87526957.json
=> Todo estaba en Kaggle: se bajo y NO se entreno.


,game_id,n_steps,winner,archivo
0,87542385,137,1,agente/ejemplos/partida_87542385.json
1,87406956,157,0,agente/ejemplos/partida_87406956.json
2,87363568,174,0,agente/ejemplos/partida_87363568.json
3,87526957,201,0,agente/ejemplos/partida_87526957.json


## Parte 3 - El agente consume el modelo evaluado, sin cambios

Primero se escribe el manifiesto con el SHA-256 de cada archivo segun el indice de Kaggle y se verifica (lo mismo que hace el agente al arrancar en la PC). Despues, en cada estado de cada partida de ejemplo, se compara:

- la probabilidad de la herramienta `estimar_probabilidad` (desde el JSON crudo), con
- la del modelo final sobre la **tabla de estados** de la Etapa 2 (misma partida, mismo paso).

Si el agente usara otra funcion de caracteristicas, otro modelo u otro orden de columnas, estas probabilidades no coincidirian.

In [29]:
# PARTE 3 - VERIFICACION CON PARTIDAS REALES
# PROPOSITO: comprobar que el agente da la misma probabilidad que el modelo final en cada estado.
# ESPERADO: una fila por partida con diferencia maxima < 1e-6 y todos los resultados verificados.
from ptcg_agente import herramientas as hz
from ptcg_agente.modelo import ModeloAgente
from ptcg_agente.registro import Registro

artefactos.preparar(RESULTADOS)
MODELO = ModeloAgente.cargar(RESULTADOS)
REGISTRO = Registro("cuaderno-07", MODELO.modelo_sha256, MODELO.commit,
                    carpeta=RESULTADOS / "agente" / "registros")
print("modelo:", MODELO.ficha["modelo"], "| SHA-256:", MODELO.modelo_sha256[:16], "| commit:", MODELO.commit)

filas = []
for g, ruta in zip(IDS, RUTAS_EJEMPLOS):
    archivo = str(RESULTADOS / ruta)
    tabla = pd.read_parquet(DATA_DIR / f"estados_{DIA_EJEMPLOS}.parquet", filters=[("game_id", "==", g)])
    tabla = tabla[(tabla["setup"] == 0) & (tabla["decided"] == 0)].sort_values("step")
    p_tabla = MODELO.predictor.probabilidades_tabla(tabla)
    diferencias, aprobados = [], 0
    for paso, p_ref in zip(tabla["step"], p_tabla):
        s = hz.llamar(MODELO, "estimar_probabilidad", {"archivo": archivo, "paso": int(paso)})
        diferencias.append(abs(s["probabilidad"] - p_ref))
        aprobados += s["verificacion"]["aprobado"]
    curva = hz.llamar(MODELO, "evaluar_partida", {"archivo": archivo}, REGISTRO, agente="cuaderno")
    filas.append({"partida": g, "estados": len(tabla), "dif_maxima": max(diferencias),
                  "verificados": f"{aprobados}/{len(tabla)}",
                  "p_final_jugador0": curva["p_final_jugador0"], "ganador_real": curva["ganador_real"]})
resumen = pd.DataFrame(filas)
display(resumen)
assert (resumen["dif_maxima"] < 1e-6).all(), "el agente no reproduce la probabilidad del modelo final"
assert all(f["verificados"].split("/")[0] == f["verificados"].split("/")[1] for f in filas)
print("OK: el agente reproduce el modelo final en", int(resumen["estados"].sum()), "estados reales")

ya local: modelo_final/b4_victoria.joblib
ya local: modelo_final/ficha.json
ya local: agente/dominio.json
ya local: evaluacion_final/test/metricas_A.csv
ya local: evaluacion_final/test/metricas_B.csv
ya local: evaluacion_final/test/metricas_C.csv
ya local: evaluacion_final/veredicto.json
modelo: B4 HistGradientBoostingClassifier (trial 20) | SHA-256: 16e7f812ba602ce3 | commit: 428d028


,partida,estados,dif_maxima,verificados,p_final_jugador0,ganador_real
0,87542385,130,0.0,130/130,0.0078,1
1,87406956,151,0.0,151/151,0.9558,0
2,87363568,166,0.0,166/166,0.9971,0
3,87526957,195,0.0,195/195,0.9996,0


OK: el agente reproduce el modelo final en 642 estados reales


## Parte 4 - Herramientas en accion, casos limite y registro

Cada llamada pasa por el verificador y queda en el registro JSONL (`/content/resultados/agente/registros/`).

In [30]:
# PARTE 4 - DEMOSTRACION DE LAS HERRAMIENTAS
# PROPOSITO: ver la salida de cada herramienta sobre una partida real y sobre entradas limite.
# ESPERADO: textos de explicacion, revision de dominio, comparacion y errores controlados.
import copy

ARCHIVO = str(RESULTADOS / RUTAS_EJEMPLOS[1])
partida = json.loads(Path(ARCHIVO).read_text(encoding="utf-8"))
curva = hz.llamar(MODELO, "evaluar_partida", {"archivo": ARCHIVO}, REGISTRO, agente="cuaderno")
print("Turnos (probabilidad del jugador 0):", [(t["turno"], t["p_jugador0"]) for t in curva["turnos"]])
print("Puntos de giro:", curva["puntos_de_giro"])

medio = next(c["paso"] for c in curva["curva"] if c["etapa"] == "medio")
temprano = curva["curva"][0]["paso"]
for paso in (temprano, medio):
    e = hz.llamar(MODELO, "explicar_prediccion", {"archivo": ARCHIVO, "paso": paso, "top": 3}, REGISTRO, agente="cuaderno")
    print(f"\npaso {paso} (turno {e['turno']}, {e['etapa']}):", e["texto"])
    print("  advertencias:", [a["codigo"] for a in e["advertencias"]], "| verificacion:", e["verificacion"])

# Casos limite y entradas inesperadas
from ptcg_wp.features import acting_player
cur = copy.deepcopy(partida["steps"][medio][acting_player(partida["steps"][medio])]["observation"]["current"])
otro = copy.deepcopy(cur)
otro["players"][1 - cur["yourIndex"]]["deckCount"] = 0
print("\ncomparar_estados:", {k: v for k, v in hz.llamar(MODELO, "comparar_estados", {"estados": [
    {"etiqueta": "real", "observacion": cur}, {"etiqueta": "rival sin mazo", "observacion": otro}]},
    REGISTRO, agente="cuaderno").items() if k in ("mejor", "mejor_texto")})
raro = copy.deepcopy(cur)
raro["players"][cur["yourIndex"]]["active"][0]["id"] = 999999
raro["players"][cur["yourIndex"]]["deckCount"] = 300
print("revisar_dominio (carta y mazo imposibles):",
      hz.llamar(MODELO, "revisar_dominio", {"observacion": raro}, REGISTRO, agente="cuaderno")["resumen"])
# Preparacion: primer paso real en turno 0 con observacion del jugador que actua
# (el paso 0 de una repeticion puede no tener jugador activo: eso es un error de entrada, no preparacion)
paso_prep = next(i for i, p in enumerate(partida["steps"]) if acting_player(p) is not None
                 and (p[acting_player(p)]["observation"].get("current") or {}).get("turn") == 0)
prep = hz.llamar(MODELO, "estimar_probabilidad", {"archivo": ARCHIVO, "paso": paso_prep}, REGISTRO, agente="cuaderno")
print(f"preparacion (paso {paso_prep}):", prep["probabilidad"], [a["codigo"] for a in prep["advertencias"]])
sin_activo = hz.llamar(MODELO, "estimar_probabilidad", {"archivo": ARCHIVO, "paso": 0}, REGISTRO, agente="cuaderno")
print("paso 0:", sin_activo.get("error") or [a["codigo"] for a in sin_activo["advertencias"]])
print("entrada mal formada:", hz.llamar(MODELO, "estimar_probabilidad", {"observacion": {"turn": 3}},
                                         REGISTRO, agente="cuaderno")["error"])

lineas = REGISTRO.archivo.read_text(encoding="utf-8").splitlines()
print(f"\nRegistro: {REGISTRO.archivo} ({len(lineas)} invocaciones). Ultima:")
print(json.dumps({k: v for k, v in json.loads(lineas[-1]).items() if k != "salida"}, indent=1, ensure_ascii=False))

Turnos (probabilidad del jugador 0): [(1, 0.5552), (2, 0.5238), (3, 0.8188), (4, 0.7218), (5, 0.7379), (6, 0.2672), (7, 0.9038), (8, 0.6882), (9, 0.9558)]
Puntos de giro: [{'paso': 98, 'turno': 6, 'de': 0.433, 'a': 0.2672, 'cambio': -0.1658, 'cruza_50': False}, {'paso': 99, 'turno': 7, 'de': 0.2672, 'a': 0.5772, 'cambio': 0.31, 'cruza_50': True}, {'paso': 122, 'turno': 8, 'de': 0.9038, 'a': 0.7188, 'cambio': -0.185, 'cruza_50': False}, {'paso': 145, 'turno': 9, 'de': 0.4656, 'a': 0.6208, 'cambio': 0.1552, 'cruza_50': True}, {'paso': 151, 'turno': 9, 'de': 0.5005, 'a': 0.6832, 'cambio': 0.1827, 'cruza_50': False}]

paso 5 (turno 1, inicio): Probabilidad de victoria: 45.5 %. Lo que mas la sube: cartas en el descarte del jugador = 0 (+8.6 puntos); PV totales en juego del rival = 110 (+6.0 puntos); cartas en el mazo del rival = 46 (+4.4 puntos). Lo que mas la baja: cartas en el descarte del rival = 0 (-9.5 puntos); numero de turno = 1 (-7.1 puntos); cartas en el mazo del jugador = 46 (-5.3

## Uso local con Claude Desktop o Claude Code

El agente se usa en la PC, no en Colab. Pasos completos en `docx/05AmbienteDeDesarrollo/GuiaDelAgente.md`:

1. Instalar `requirements.txt` en un entorno de Python 3.11 o superior.
2. Bajar y verificar los archivos del agente: `python -m ptcg_agente.artefactos` (con `PYTHONPATH=src` y el token de Kaggle).
3. **Claude Code:** abrir el repositorio; el archivo `.mcp.json` registra el servidor `ptcg-victoria`.
4. **Claude Desktop:** agregar el servidor en `claude_desktop_config.json` con rutas absolutas.
5. Pedirle a Claude, por ejemplo: *"Usa ptcg-victoria para analizar artefactos_agente/agente/ejemplos/partida_XXXX.json: ¿quien iba ganando en cada turno y por que?"*